In [1]:
!pip install ultralytics
!pip install supervision

In [6]:
from ultralytics import YOLO
import supervision as sv
import cv2
model = YOLO('yolov8n.pt')
path = "/content/project.mp4"
cap = cv2.VideoCapture(path)
track = sv.ByteTrack()
box = sv.BoxAnnotator()

In [ ]:
from google.colab.patches import cv2_imshow
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
fps = int(cap.get(cv2.CAP_PROP_FPS))


output_path = 'output.avi'
fourcc = cv2.VideoWriter_fourcc(*'XVID')
out = cv2.VideoWriter(output_path, fourcc, fps, (width,height))

while True:
    ret, frame = cap.read()
    if not ret:
        break
    results = model(frame)[0]
    detections = sv.Detections.from_ultralytics(results)
    tracked_detections = track.update_with_detections(detections)
    frame = box.annotate(scene=frame, detections=tracked_detections)
    for box_item in range(len(tracked_detections)):
      x1,y1,x2,y2 = tracked_detections.xyxy[box_item]
      class_id = tracked_detections.class_id[box_item]
      confidence = tracked_detections.confidence[box_item]
      tracked_id = tracked_detections.tracker_id[box_item]
      color = (0,255,0) if class_id == 0 else (0,0,255)
      label = f"{model.names[class_id]} , {confidence:.2f}, id:{tracked_id}"
      cv2.putText(frame, label, (int(x1), int(y1)-10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color,2)
    out.write(frame)
    cv2_imshow(frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break
cap.release()
out.release()
cv2.destroyAllWindows()